# NDT-Transformer on KITTI: experiments

Runtime -> Change runtime type -> **T4 GPU** (or better).

**One experiment = one NDT variant + one training run.** Set both in cell 1 and run every cell from top to bottom.
Every cell skips work that is already done, so the same order works for a new experiment, after a disconnect, or in a new session:

- **NDT variant**: the preprocessing + G parameters. Saved in `MyDrive/lpr/ndt/<variant>/` with its `config.json`, the cells, per-scan stats and a preview picture. A variant with the same parameters is never built twice.
- **Run**: the training parameters. Saved in `MyDrive/lpr/runs/<run>/` with `config.json` (parameters, NDT variant, GPU, code version, status), `metrics.csv`, a checkpoint per epoch and the final evaluation. An unfinished run with the same parameters is resumed; finished runs are never overwritten.
- `MyDrive/lpr/runs_summary.csv`: one row per run (cell 8).

Names are `<date>_<tag>_<hash>`: the hash is computed from the parameters.

## 1. Experiment settings

In [ ]:
REPO_URL = "https://github.com/dov85/lidar-place-recognition"
DRIVE = "/content/drive/MyDrive/lpr"
SEQ = "00"

# NDT variant: only what differs from the defaults (src/lpr/experiments.py, NDT_DEFAULTS):
#   crop="square", half_size=40.0, ground="patchwork", k=2000, over=0.10, radius=0.8,
#   min_pts=5, knn=8, cov_eps=1e-3
NDT_TAG = "baseline"
NDT_PARAMS = dict()                 # e.g. dict(crop="circle", radius=1.0)

# Training run: only what differs from the defaults (TRAIN_DEFAULTS):
#   pos_thr=5.0, neg_thr=7.0, epochs=20, lr=1e-5, milestones=[9, 15], batch_queries=2,
#   positives=2, negatives=18, margin1=0.5, margin2=0.2, hard_neg_after_epoch=5, amp=True, seed=0
RUN_TAG = "baseline"
TRAIN_PARAMS = dict()               # e.g. dict(lr=5e-6, negatives=12)

KITTI = "/content/kitti"                       # raw scans + poses (temporary)
RAW_TAR = f"{DRIVE}/kitti_seq00_raw.tar"       # backup of the raw scans of sequence 00

## 2. Drive, GPU, code, poses

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, subprocess, importlib
os.makedirs(DRIVE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

url = REPO_URL
try:  # private repo: store a token in Colab Secrets as GITHUB_TOKEN
    from google.colab import userdata
    url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass
if not os.path.exists("/content/lpr"):
    subprocess.run(["git", "clone", "-q", url, "/content/lpr"], check=True)
else:
    subprocess.run(["git", "-C", "/content/lpr", "pull", "-q"], check=True)
%cd /content/lpr
!git log --oneline -1
!bash scripts/setup_third_party.sh
!pip install -q -r requirements.txt
if not os.path.exists(f"{KITTI}/poses/{SEQ}.txt"):
    !python scripts/download_kitti.py --out {KITTI} --seqs {SEQ} --poses_only

sys.path.insert(0, "/content/lpr/src")
import lpr.experiments as ex
importlib.reload(ex)

### 2b. One time only: move the Stage 1 results into the new layout
The first run (`runs/ndt_kitti00`) and its NDT cells (`ndt_kitti00_patchwork.tar`) become the `baseline` variant and run.
Run once with `APPLY = False` to see the plan, then with `APPLY = True`. Afterwards this cell does nothing.

In [ ]:
APPLY = False
apply_flag = "--apply" if APPLY else ""
if os.path.exists(f"{DRIVE}/runs/ndt_kitti00"):
    !python scripts/migrate_stage1.py --drive {DRIVE} {apply_flag}
else:
    print("nothing to migrate")

## 3. Names of the NDT variant and the run

In [ ]:
VARIANT, NDT_FULL, ndt_exists = ex.resolve_ndt_variant(DRIVE, dict(NDT_PARAMS, seq=SEQ), NDT_TAG)
VDIR = f"{DRIVE}/ndt/{VARIANT}"          # permanent copy on Drive
NDT = f"/content/ndt/{VARIANT}"          # working copy on this machine
CELLS_TAR = f"{VDIR}/cells_{SEQ}.tar"

RUN, TRAIN_FULL, run_state = ex.resolve_run(DRIVE, dict(TRAIN_PARAMS, seq=SEQ), VARIANT, RUN_TAG)
RUN_DIR = f"{DRIVE}/runs/{RUN}"

print("NDT variant:", VARIANT, "(exists on Drive)" if ndt_exists else "(new: will be built in cell 4)")
print(json.dumps(NDT_FULL))
print("run:        ", RUN, f"({run_state})")
print(json.dumps(TRAIN_FULL))

## 4. NDT cells: restore from Drive, or build the new variant
Building a variant restores the raw scans from `RAW_TAR` (or downloads them), runs preprocessing + G on every scan
(about 0.5 s per scan on 2 CPU cores), and saves everything to `VDIR`. `config.json` is written last, so a variant
on Drive is always complete.

In [ ]:
from IPython.display import Image, display
ndt_args = " ".join(f"--{k} {v}" for k, v in NDT_FULL.items() if k != "seq")

if os.path.exists(f"{NDT}/{SEQ}/000000.bin"):
    print("NDT cells already on this machine:", NDT)
elif ndt_exists:
    os.makedirs(NDT, exist_ok=True)
    !tar -xf {CELLS_TAR} -C {NDT}
    !cp {VDIR}/config.json {NDT}/config.json
else:
    if not os.path.exists(f"{KITTI}/sequences/{SEQ}/velodyne"):
        if os.path.exists(RAW_TAR):
            !mkdir -p {KITTI} && tar -xf {RAW_TAR} -C {KITTI}
        else:
            !python scripts/download_kitti.py --out {KITTI} --seqs {SEQ} --threads 48
    os.makedirs(VDIR, exist_ok=True)
    !python scripts/visualize_ndt.py --kitti_root {KITTI} --seq {SEQ} --idx 0 --variant {VARIANT} --out {VDIR}/preview_{SEQ}.png {ndt_args}
    display(Image(f"{VDIR}/preview_{SEQ}.png"))
    !python scripts/extract_ndt.py --kitti_root {KITTI} --seq {SEQ} --out_dir {NDT} --variant {VARIANT} {ndt_args}
    !tar -cf {CELLS_TAR} -C {NDT} {SEQ} {SEQ}_stats.csv config.json
    !cp {NDT}/{SEQ}_stats.csv {VDIR}/stats_{SEQ}.csv
    !cp {NDT}/config.json {VDIR}/config.json
!ls {NDT}/{SEQ} | wc -l
if os.path.exists(f"{VDIR}/config.json"):
    print(json.dumps(json.load(open(f"{VDIR}/config.json")), indent=1))

## 5. Training tuples (frames 0-2999 of the sequence)

In [ ]:
pos_thr, neg_thr = TRAIN_FULL["pos_thr"], TRAIN_FULL["neg_thr"]
!python scripts/make_kitti_tuples.py --kitti_root {KITTI} --seq {SEQ} --ndt_dir {NDT} --out {NDT}/train_{SEQ}.pickle --pos_thr {pos_thr} --neg_thr {neg_thr}

## 6. Optional: timing and memory test (30 iterations)
Seconds per iteration x 1500 is roughly the time of one epoch (more from epoch 7, when hard negatives start).

In [ ]:
t = TRAIN_FULL
train_args = (f"--ndt_dir {NDT} --kitti_root {KITTI} --train_pickle {NDT}/train_{SEQ}.pickle "
              f"--ndt_variant {VARIANT} --seq {SEQ} --epochs {t['epochs']} --lr {t['lr']} "
              f"--milestones {' '.join(map(str, t['milestones']))} --batch_queries {t['batch_queries']} "
              f"--positives {t['positives']} --negatives {t['negatives']} --margin1 {t['margin1']} "
              f"--margin2 {t['margin2']} --hard_neg_after_epoch {t['hard_neg_after_epoch']} --seed {t['seed']}"
              + (" --amp" if t["amp"] else ""))
RUN_SMOKE = False
if RUN_SMOKE:
    !rm -rf /content/smoke
    !python scripts/train_ndt_kitti.py {train_args} --log_dir /content/smoke --epochs 1 --max_iters 30 --eval_every 0 2>&1 | grep -v Warning

## 7. Training (resumes automatically after a disconnect)

In [ ]:
print("run:", RUN_DIR, f"({run_state})")
!python scripts/train_ndt_kitti.py {train_args} --log_dir {RUN_DIR} 2>&1 | grep -v Warning

## 8. Evaluation of the last epoch, and the table of all runs

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
last = sorted(f for f in os.listdir(RUN_DIR) if f.startswith("epoch_"))[-1]
cmd = ["python", "scripts/eval_ndt_kitti.py", "--ckpt", f"{RUN_DIR}/{last}", "--ndt_dir", NDT,
       "--kitti_root", KITTI, "--seq", SEQ, "--out", f"{RUN_DIR}/eval_{SEQ}.json"]
res = subprocess.run(cmd, capture_output=True, text=True, cwd="/content/lpr")
print("checkpoint:", last)
print(res.stdout if res.returncode == 0 else res.stderr[-3000:])

m = pd.read_csv(f"{RUN_DIR}/metrics.csv").drop_duplicates("epoch", keep="last")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(m.epoch, m.train_loss, marker="o"); ax[0].set_title("training loss"); ax[0].set_xlabel("epoch")
ax[1].plot(m.epoch, m["R@1"], marker="o", label="R@1"); ax[1].plot(m.epoch, m["R@1%"], marker="o", label="R@1%")
ax[1].set_title(f"KITTI {SEQ} recall (%)  {RUN}"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.show()

In [ ]:
!python scripts/summarize_runs.py --drive {DRIVE} > /dev/null
display(pd.read_csv(f"{DRIVE}/runs_summary.csv"))